# Continuous-type case study: the value of joint design

$n$ customer types with delay sensitivities at evenly spaced quantiles of a
uniform distribution, each arriving at rate $\lambda/n$. Service values and
service-time moments come from the SGD experiment. Four designs are compared:

| Design | Service depth | Scheduling |
|---|---|---|
| Joint | type-dependent, from $\{d_0,\dots,d_5\}$ | nonpreemptive priority in $c\mu$ order |
| Depth only | type-dependent | FCFS |
| Priority only | one common depth for all types | nonpreemptive priority in $c\mu$ order |
| Neither | one common depth for all types | FCFS |

Given the depths, the $c\mu$ order $\theta_i/m_i$ is optimal among
nonpreemptive priority rules for any service distribution. Type-dependent depth
assignments are optimized by exhaustive enumeration of assignments that are
non-increasing in $\theta$ (five thresholds), followed by an unrestricted local
search to confirm that no other assignment does better. Exact stationary M/G/1
formulas throughout; no simulation.

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sgd_calibration import load_calibration

cal = load_calibration(proxy="cost")
VALUES, MEANS, SECONDS = cal["values"], cal["means"], cal["seconds"]
EFFORTS = list(cal["efforts"])
print("values ", np.round(VALUES, 3))
print("means  ", np.round(MEANS, 4))
print("E[S^2] ", np.round(SECONDS, 4))

## Parameters

In [ ]:
N = 40                                   # number of types
THETA_LO, THETA_HI = 0.005, 0.05         # uniform delay-sensitivity range
THETA = np.linspace(THETA_LO, THETA_HI, N)   # index 0 = least sensitive, N-1 = most sensitive
LAMBDAS = np.round(np.arange(0.30, 1.1501, 0.02), 4)
LAMBDA_STAR = 1.00                       # load at which the allocation is displayed
OUTPUT = Path("results") / "continuous_type_load"

# Depth index convention: -1 = d_0 (no service), 0..4 = none, low, medium, high, xhigh.
M_EXT = np.r_[MEANS, 0.0]; B_EXT = np.r_[SECONDS, 0.0]; V_EXT = np.r_[VALUES, 0.0]   # index -1 -> 0

## Exact stationary evaluation

For served types with loads $\rho_i=(\lambda/n)m_i$ and $R=\tfrac12\sum_i(\lambda/n)b_i$:
FCFS gives $w_i=m_i+R/(1-\rho)$; nonpreemptive priority gives
$w_i=m_i+R/((1-\sigma_i^{+})(1-\sigma_i^{+}-\rho_i))$ where $\sigma_i^{+}$ is the
load of types with higher priority (Cobham). Types at $d_0$ have $w_i=0$.

In [ ]:
def evaluate(depths, lam, rule):
    """Welfare and sojourn times for one assignment (array of -1..4)."""
    depths = np.asarray(depths)
    served = depths >= 0
    li = lam / N
    m, b, v = M_EXT[depths], B_EXT[depths], V_EXT[depths]
    rho = li * m
    total = rho.sum()
    if total >= 1.0:
        return -np.inf, None
    R = 0.5 * li * b.sum()
    w = np.zeros(N)
    if rule == "FCFS":
        w[served] = m[served] + R / (1.0 - total)
    else:
        idx = np.flatnonzero(served)
        order = idx[np.argsort(-(THETA[idx] / m[idx]), kind="stable")]   # highest theta/m first
        sig = np.cumsum(rho[order]); sig_prev = sig - rho[order]
        w[order] = m[order] + R / ((1.0 - sig_prev) * (1.0 - sig))
    W = (li * (v - THETA * w))[served].sum()
    return W, w


def evaluate_monotone_batch(counts, lam, rule):
    """Vectorized welfare for monotone assignments given by level counts.

    counts: (K, 6) numbers of types at depths xhigh, high, medium, low, none, d_0,
    in that order from the least to the most sensitive type. Returns (K,) welfare.
    """
    K = counts.shape[0]
    cum = np.cumsum(counts, axis=1)[:, :-1]                     # (K, 5) boundaries
    pos = np.arange(N)
    level = (pos[None, :] >= cum[:, :, None]).sum(axis=1)       # 0 = xhigh ... 5 = d_0
    depth = 4 - level                                            # 4..-1
    m, b, v = M_EXT[depth], B_EXT[depth], V_EXT[depth]           # (K, N)
    li = lam / N
    rho = li * m
    total = rho.sum(axis=1)
    R = 0.5 * li * b.sum(axis=1)
    served = depth >= 0
    if rule == "FCFS":
        wq = (R / (1.0 - total))[:, None] * np.ones((1, N))
    else:
        # monotone depths => cmu order is decreasing theta, i.e. positions N-1, N-2, ...
        sig = np.cumsum(rho[:, ::-1], axis=1)[:, ::-1]            # load of self + more sensitive types
        sig_prev = sig - rho
        wq = R[:, None] / ((1.0 - sig_prev) * (1.0 - sig))
    w = np.where(served, m + wq, 0.0)
    W = (li * (v - THETA[None, :] * w) * served).sum(axis=1)
    return np.where(total < 1.0, W, -np.inf)


def compositions(n, parts):
    """All nonnegative integer vectors of length `parts` summing to n."""
    out = []
    for bars in itertools.combinations(range(n + parts - 1), parts - 1):
        prev, row = -1, []
        for bb in bars:
            row.append(bb - prev - 1); prev = bb
        row.append(n + parts - 2 - prev)
        out.append(row)
    return np.array(out, dtype=np.int16)


COUNTS = compositions(N, 6)
print(f"{len(COUNTS):,} monotone assignments enumerated")


def counts_to_depths(c):
    return np.repeat(np.arange(4, -2, -1), c)


def best_monotone(lam, rule, chunk=200_000):
    best_W, best_c = -np.inf, None
    for s in range(0, len(COUNTS), chunk):
        W = evaluate_monotone_batch(COUNTS[s:s + chunk], lam, rule)
        j = int(np.argmax(W))
        if W[j] > best_W:
            best_W, best_c = float(W[j]), COUNTS[s + j]
    return best_W, counts_to_depths(best_c)


def local_search(depths, lam, rule):
    """Unrestricted coordinate descent from a starting assignment; returns the improvement found."""
    d = depths.copy(); W0, _ = evaluate(d, lam, rule); Wc = W0
    improved = True
    while improved:
        improved = False
        for i in range(N):
            cur = d[i]
            for k in range(-1, 5):
                if k == cur:
                    continue
                d[i] = k; Wk, _ = evaluate(d, lam, rule)
                if Wk > Wc + 1e-12:
                    Wc, cur, improved = Wk, k, True
            d[i] = cur
    return Wc - W0, d


def best_common(lam, rule):
    best = (-np.inf, None)
    for k in range(-1, 5):
        W, _ = evaluate(np.full(N, k), lam, rule)
        if W > best[0]:
            best = (W, k)
    return best

## Sweep over the arrival rate

In [ ]:
results = {k: [] for k in ("joint", "depth", "priority", "neither")}
alloc = {}
max_nonmonotone_gain = 0.0
for lam in LAMBDAS:
    Wj, dj = best_monotone(lam, "NP")
    Wd, dd = best_monotone(lam, "FCFS")
    gain_j, _ = local_search(dj, lam, "NP")
    gain_d, _ = local_search(dd, lam, "FCFS")
    max_nonmonotone_gain = max(max_nonmonotone_gain, gain_j, gain_d)
    Wp, kp = best_common(lam, "NP")
    Wn, kn = best_common(lam, "FCFS")
    for k, W in zip(("joint", "depth", "priority", "neither"), (Wj, Wd, Wp, Wn)):
        results[k].append(max(W, 0.0))          # closing the system yields zero welfare
    alloc[lam] = dict(joint=dj, depth=dd, priority=np.full(N, kp), neither=np.full(N, kn))
W = {k: np.array(v) for k, v in results.items()}
print(f"largest welfare gain from any non-monotone deviation: {max_nonmonotone_gain:.2e}")

def label(d):
    return "d0" if d < 0 else EFFORTS[d]

def runs(d):
    out, s = [], 0
    for i in range(1, N + 1):
        if i == N or d[i] != d[s]:
            out.append(f"{label(d[s])}x{i - s}"); s = i
    return " ".join(out)

for lam in (0.7, 0.9, 1.0, 1.05, 1.1):
    i = int(np.argmin(np.abs(LAMBDAS - lam)))
    print(f"lambda={LAMBDAS[i]:.2f}: joint={W['joint'][i]:.4f} depth={W['depth'][i]:.4f} "
          f"priority={W['priority'][i]:.4f} neither={W['neither'][i]:.4f} | joint allocation: {runs(alloc[LAMBDAS[i]]['joint'])}")

## Figure 1: welfare of the four designs

In [ ]:
RED = "#c0392b"
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.labelsize": 10,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "pdf.fonttype": 42, "axes.linewidth": 0.6, "lines.linewidth": 1.5})

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(LAMBDAS, W["joint"], color="#222222", linewidth=2.2, label=r"Joint ($c\mu$ priority)")
ax.plot(LAMBDAS, W["depth"], color="#1f77b4", linewidth=2.0, linestyle="-.", label="Depth only (FCFS)")
ax.plot(LAMBDAS, W["priority"], color=RED, linewidth=2.0, linestyle="--", label=r"Priority only ($c\mu$ priority)")
ax.plot(LAMBDAS, W["neither"], color="#2ca02c", linewidth=3.0, linestyle=":", label="Neither (FCFS)")
ax.set_xlim(LAMBDAS[0], LAMBDAS[-1])
ax.set_xlabel(r"Arrival rate $\lambda$", fontsize=24)
ax.set_ylabel(r"$\mathcal{W}$", fontsize=26, rotation=0, labelpad=18)
ax.yaxis.set_label_coords(-0.2, 0.5)
ax.tick_params(labelsize=18)
ax.legend(fontsize=14, loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=2, frameon=True,
          edgecolor="#cccccc", framealpha=1.0, handlelength=1.6, borderpad=0.4, columnspacing=1.2)
ax.grid(axis="y", alpha=0.18)
fig.subplots_adjust(left=0.21, right=0.96, top=0.82, bottom=0.21)
fig.savefig(OUTPUT.with_name("continuous_type_load_welfare.png"), dpi=300)
fig.savefig(OUTPUT.with_name("continuous_type_load_welfare.pdf"))
plt.show()

## Figure 2: allocation at $\lambda^\star$ under the four designs

In [ ]:
i_star = int(np.argmin(np.abs(LAMBDAS - LAMBDA_STAR)))
lam_star = LAMBDAS[i_star]
STYLE = {"joint": dict(color="#222222", linewidth=2.4, linestyle="-", label=r"Joint ($c\mu$ priority)"),
         "depth": dict(color="#1f77b4", linewidth=2.0, linestyle="-.", label="Depth only (FCFS)"),
         "priority": dict(color=RED, linewidth=2.0, linestyle="--", label=r"Priority only ($c\mu$ priority)"),
         "neither": dict(color="#2ca02c", linewidth=3.2, linestyle=":", label="Neither (FCFS)")}
RULE = {"joint": "NP", "depth": "FCFS", "priority": "NP", "neither": "FCFS"}
soj = {}
for k in STYLE:
    d = alloc[lam_star][k]
    Wk, wk = evaluate(d, lam_star, RULE[k])
    soj[k] = (d, wk)
    print(f"lambda*={lam_star}: {k:9s} W={Wk:.4f}  depths: {runs(d)}")
print("joint sojourn non-increasing in theta among served types:",
      bool(np.all(np.diff(soj["joint"][1][soj["joint"][0] >= 0]) <= 1e-12)))

fig, ax = plt.subplots(figsize=(7.2, 4.8))
for k in ("priority", "depth", "joint", "neither"):
    ax.step(THETA, soj[k][0] + 1, where="mid", **STYLE[k])
ax.set_xlim(THETA_LO, THETA_HI)
ax.set_ylim(-0.4, 5.5)
ax.set_yticks(range(6), [f"$d_{k}$" for k in range(6)])
ax.set_xlabel(r"Delay sensitivity $\theta$", fontsize=24)
ax.tick_params(labelsize=18)
handles = [plt.Line2D([], [], **{kk: vv for kk, vv in STYLE[k].items()}) for k in ("joint", "depth", "priority", "neither")]
ax.legend(handles=handles, fontsize=14, loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=2, frameon=True,
          edgecolor="#cccccc", framealpha=1.0, handlelength=1.6, borderpad=0.4, columnspacing=1.2)
ax.grid(axis="y", alpha=0.18)
fig.subplots_adjust(left=0.13, right=0.96, top=0.82, bottom=0.21)
fig.savefig(OUTPUT.with_name("continuous_type_load_depth.png"), dpi=300)
fig.savefig(OUTPUT.with_name("continuous_type_load_depth.pdf"))
plt.show()

fig, ax = plt.subplots(figsize=(7.2, 4.8))
for k in ("priority", "depth", "joint", "neither"):
    d, w = soj[k]; served = d >= 0
    ax.plot(THETA[served], w[served], **STYLE[k])
ax.set_xlim(THETA_LO, THETA_HI)
ax.set_yscale("log")
ax.set_xlabel(r"Delay sensitivity $\theta$", fontsize=24)
ax.set_ylabel(r"$w(\theta)$", fontsize=24, rotation=0, labelpad=18)
ax.yaxis.set_label_coords(-0.17, 0.5)
ax.tick_params(labelsize=18)
ax.legend(handles=handles, fontsize=14, loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=2, frameon=True,
          edgecolor="#cccccc", framealpha=1.0, handlelength=1.6, borderpad=0.4, columnspacing=1.2)
ax.grid(axis="y", alpha=0.18)
fig.subplots_adjust(left=0.21, right=0.96, top=0.82, bottom=0.21)
fig.savefig(OUTPUT.with_name("continuous_type_load_sojourn.png"), dpi=300)
fig.savefig(OUTPUT.with_name("continuous_type_load_sojourn.pdf"))
plt.show()

## Self-checks

The batch evaluator agrees with the scalar one; a single served type reproduces the
M/G/1 FCFS formula; the two-type nonpreemptive formulas are recovered when only two
types are served.

In [ ]:
rng = np.random.default_rng(0)
for _ in range(20):
    c = rng.multinomial(N, np.full(6, 1 / 6))
    for rule in ("FCFS", "NP"):
        Wb = evaluate_monotone_batch(c[None, :], 0.8, rule)[0]
        Ws, _ = evaluate(counts_to_depths(c), 0.8, rule)
        assert np.isinf(Wb) and np.isinf(Ws) or abs(Wb - Ws) < 1e-12
d = np.full(N, -1); d[0] = 2
_, w = evaluate(d, 0.8, "NP")
li = 0.8 / N
assert abs(w[0] - (MEANS[2] + li * SECONDS[2] / (2 * (1 - li * MEANS[2])))) < 1e-12
d = np.full(N, -1); d[0], d[N - 1] = 3, 1                    # patient type high, sensitive type low
_, w = evaluate(d, 0.8, "NP")
l1 = l2 = li; m1, m2 = MEANS[1], MEANS[3]; b1, b2 = SECONDS[1], SECONDS[3]
R = 0.5 * (l1 * b1 + l2 * b2); rho1, rho2 = l1 * m1, l2 * m2
assert abs(w[N - 1] - (m1 + R / (1 - rho1))) < 1e-12                       # sensitive type has priority
assert abs(w[0] - (m2 + R / ((1 - rho1) * (1 - rho1 - rho2)))) < 1e-12
print("all checks passed")